In [1]:


from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.appName("StockoutFeatureEng").getOrCreate()

# ── Table names (as they appear in your Lakehouse) ─────────
# Update these if your table names differ
TABLE_SALES    = "sales_data"
TABLE_INV      = "inventory_data"
TABLE_OOS      = "stockout_events"

# ══════════════════════════════════════════════════════════
# LOAD FROM LAKEHOUSE TABLES
# ══════════════════════════════════════════════════════════
sales_raw = spark.read.table(TABLE_SALES)
inv_raw   = spark.read.table(TABLE_INV)
oos_raw   = spark.read.table(TABLE_OOS)

print(f"Loaded from tables:")
print(f"  {TABLE_SALES}    : {sales_raw.count():,} rows | cols: {sales_raw.columns}")
print(f"  {TABLE_INV}  : {inv_raw.count():,} rows | cols: {inv_raw.columns}")
print(f"  {TABLE_OOS} : {oos_raw.count():,} rows | cols: {oos_raw.columns}")

# ══════════════════════════════════════════════════════════
# CLEAN: SALES
# R1 - Remove duplicate (date, sku_id) rows
# R2 - Null units_sold → 0  (no record = no sale)
# R3 - Negative units_sold → filter out  (data entry error)
# R4 - unit_price must be > 0
# R5 - Standardise string casing
# ══════════════════════════════════════════════════════════
sales = (
    sales_raw
    .withColumn("date", F.to_date("date"))
    .dropDuplicates(["date", "sku_id"])
    .fillna({"units_sold": 0, "promo_flag": 0, "is_festival": 0, "discount_pct": 0.0})
    .filter((F.col("units_sold") >= 0) & (F.col("unit_price") > 0))
    .withColumn("sku_id",   F.trim(F.upper("sku_id")))
    .withColumn("category", F.initcap("category"))
    .withColumn("store_id", F.trim("store_id"))
)

print(f"\nSales after cleaning   : {sales.count():,} rows")
sales.printSchema()

# ══════════════════════════════════════════════════════════
# CLEAN: INVENTORY
# R1 - Cast date
# R2 - Filter negative stock_on_hand  (impossible value)
# R3 - Forward-fill null stock_on_hand per SKU
# R4 - Rename replenishment_qty_ordered → replenishment_qty
# R5 - Rename below_safety_stock → below_safety_stock_flag
# R6 - Recalculate days_of_supply cleanly (fixes 0-demand NaN)
# ══════════════════════════════════════════════════════════
inv = (
    inv_raw
    .withColumn("date", F.to_date("date"))
    .filter(F.col("stock_on_hand").isNull() | (F.col("stock_on_hand") >= 0))
)

# Forward-fill null stock_on_hand per SKU
ffill_w = Window.partitionBy("sku_id").orderBy("date").rowsBetween(Window.unboundedPreceding, 0)
inv = (
    inv
    .withColumn("stock_on_hand",
                F.last("stock_on_hand", ignorenulls=True).over(ffill_w))
    .fillna({"stock_on_hand": 0})
    .withColumnRenamed("replenishment_qty_ordered", "replenishment_qty")
    .withColumnRenamed("below_safety_stock",        "below_safety_stock_flag")
    .withColumn("days_of_supply",
                F.when(F.col("stock_on_hand") > 0,
                       F.col("stock_on_hand") / F.greatest(F.col("safety_stock"), F.lit(1))
                ).otherwise(F.lit(0.0)))
    .withColumn("sku_id", F.trim(F.upper("sku_id")))
)

print(f"Inventory after cleaning: {inv.count():,} rows")
inv.printSchema()

# ══════════════════════════════════════════════════════════
# CLEAN: STOCKOUT EVENTS
# R1 - Cast dates
# R2 - Drop rows where start_date > end_date (corrupt)
# R3 - Sentinel -1 for open/unresolved stockouts
# ══════════════════════════════════════════════════════════
oos = (
    oos_raw
    .withColumn("stockout_start_date", F.to_date("stockout_start_date"))
    .withColumn("stockout_end_date",   F.to_date("stockout_end_date"))
    .filter(
        F.col("stockout_end_date").isNull() |
        (F.col("stockout_start_date") <= F.col("stockout_end_date"))
    )
    .fillna({"stockout_duration_days": -1})
)

print(f"Stockout events clean  : {oos.count():,} rows")

# ══════════════════════════════════════════════════════════
# SAVE CLEAN TABLES TO LAKEHOUSE
# ══════════════════════════════════════════════════════════
sales.write.mode("overwrite").format("delta").saveAsTable("sales_clean")
inv.write.mode("overwrite").format("delta").saveAsTable("inventory_clean")
oos.write.mode("overwrite").format("delta").saveAsTable("stockout_events_clean")

print("\nClean tables written:")
print("  → sales_clean")
print("  → inventory_clean")
print("  → stockout_events_clean")

# ══════════════════════════════════════════════════════════
# FEATURE ENGINEERING
# Join inventory_clean + sales_clean → build ml_features table
# ══════════════════════════════════════════════════════════
base = inv.join(
    sales.select("date","sku_id","sku_name","category",   # category & sku_name live in sales, not inventory
                 "units_sold","promo_flag",
                 "is_weekend","is_festival","revenue","discount_pct","shelf_life_days"),
    on=["date","sku_id"],
    how="left"
).fillna({
    "units_sold":   0,
    "promo_flag":   0,
    "revenue":      0.0,
    "is_weekend":   0,
    "is_festival":  0,
    "discount_pct": 0.0
})

# ── Rolling demand features ────────────────────────────────
for days, alias in [(7,"7d"),(14,"14d"),(30,"30d")]:
    w = Window.partitionBy("sku_id").orderBy("date").rowsBetween(-(days-1), 0)
    base = (
        base
        .withColumn(f"avg_sales_{alias}", F.avg("units_sold").over(w))
        .withColumn(f"sum_sales_{alias}", F.sum("units_sold").over(w))
        .withColumn(f"max_sales_{alias}", F.max("units_sold").over(w))
    )

w7 = Window.partitionBy("sku_id").orderBy("date").rowsBetween(-6, 0)
base = (
    base
    .withColumn("std_sales_7d",      F.stddev("units_sold").over(w7))
    .withColumn("promo_count_7d",    F.sum("promo_flag").over(w7))
    .withColumn("festival_count_7d", F.sum("is_festival").over(w7))
)

# ── Inventory ratio features ───────────────────────────────
base = (
    base
    .withColumn("stock_to_safety_ratio",
                F.col("stock_on_hand") / F.greatest(F.col("safety_stock"), F.lit(1)))
    .withColumn("stock_vs_reorder",
                F.col("stock_on_hand") / F.greatest(F.col("reorder_point"), F.lit(1)))
    .withColumn("projected_cover_days",
                F.when(F.col("avg_sales_7d") > 0,
                       F.col("stock_on_hand") / F.col("avg_sales_7d"))
                 .otherwise(F.col("stock_on_hand").cast("double")))
)

# ── Lag features (1, 3, 7 days) ───────────────────────────
lag_w = Window.partitionBy("sku_id").orderBy("date")
for lag in [1, 3, 7]:
    base = (
        base
        .withColumn(f"units_sold_lag{lag}", F.lag("units_sold",    lag).over(lag_w))
        .withColumn(f"stock_lag{lag}",      F.lag("stock_on_hand", lag).over(lag_w))
        .withColumn(f"soh_delta_{lag}d",
                    F.col("stock_on_hand") - F.lag("stock_on_hand", lag).over(lag_w))
    )

# ── Calendar features ──────────────────────────────────────
base = (
    base
    .withColumn("day_of_week",  F.dayofweek("date"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("month",        F.month("date"))
    .withColumn("quarter",      F.quarter("date"))
    .withColumn("is_month_end", F.when(F.dayofmonth("date") >= 28, 1).otherwise(0))
)

# ── Category encoding ──────────────────────────────────────
base = base.withColumn("category_code",
    F.when(F.col("category") == "Beverages",     1)
     .when(F.col("category") == "Dairy",         2)
     .when(F.col("category") == "Snacks",        3)
     .when(F.col("category") == "Personal Care", 4)
     .when(F.col("category") == "Household",     5)
     .when(F.col("category") == "Staples",       6)
     .when(F.col("category") == "Frozen Foods",  7)
     .otherwise(0)
)

base = base.withColumn(
    "is_perishable",
    F.when(F.col("category").isin("Dairy","Frozen Foods"), 1).otherwise(0)
)

# ── Target variable ────────────────────────────────────────
base = base.withColumn(
    "days_until_stockout",
    F.least(
        F.greatest(
            F.round(
                F.when(F.col("avg_sales_7d") > 0,
                       F.col("stock_on_hand") / F.col("avg_sales_7d"))
                 .otherwise(F.lit(90.0))
            ),
            F.lit(0)
        ),
        F.lit(90)
    ).cast(IntegerType())
)

# ── Select final columns & drop rows with no target ────────
FEATURE_COLS = [
    "date","sku_id","sku_name","category","store_id",
    "stock_on_hand","safety_stock","reorder_point",
    "below_safety_stock_flag","days_of_supply",
    "stock_to_safety_ratio","stock_vs_reorder","projected_cover_days",
    "replenishment_qty","lead_time_days",
    "avg_sales_7d","avg_sales_14d","avg_sales_30d",
    "sum_sales_7d","max_sales_7d","std_sales_7d",
    "promo_count_7d","festival_count_7d",
    "units_sold_lag1","units_sold_lag3","units_sold_lag7",
    "stock_lag1","stock_lag3","stock_lag7",
    "soh_delta_1d","soh_delta_3d","soh_delta_7d",
    "day_of_week","week_of_year","month","quarter",
    "is_weekend","is_festival","is_month_end",
    "category_code","is_perishable","shelf_life_days",
    "days_until_stockout",
]

features_df = base.select(*FEATURE_COLS).dropna(subset=["days_until_stockout","avg_sales_7d"])

# ══════════════════════════════════════════════════════════
# SAVE FEATURE TABLE TO LAKEHOUSE
# ══════════════════════════════════════════════════════════
features_df.write.mode("overwrite").format("delta").saveAsTable("ml_features")

print(f"\nFeature table written  : ml_features")
print(f"  Rows    : {features_df.count():,}")
print(f"  Columns : {len(FEATURE_COLS)}")

# Null check
null_counts = {
    c: features_df.filter(F.col(c).isNull()).count()
    for c in FEATURE_COLS
}
nulls_found = {k: v for k, v in null_counts.items() if v > 0}
if nulls_found:
    print(f"\n  Nulls remaining in feature table: {nulls_found}")
else:
    print("\n  No nulls in ml_features. Ready for ML training.")

StatementMeta(, 0b682a63-33c1-487b-a935-a7fbc24c9af5, 3, Finished, Available, Finished, False)

Loaded from tables:
  sales_data    : 27,450 rows | cols: ['date', 'sku_id', 'sku_name', 'category', 'brand', 'unit', 'store_id', 'units_sold', 'unit_price', 'discount_pct', 'revenue', 'promo_flag', 'is_weekend', 'is_festival', 'shelf_life_days']
  inventory_data  : 27,450 rows | cols: ['date', 'sku_id', 'store_id', 'stock_on_hand', 'safety_stock', 'reorder_point', 'replenishment_qty_ordered', 'lead_time_days', 'days_of_supply', 'below_safety_stock']
  stockout_events : 375 rows | cols: ['sku_id', 'sku_name', 'category', 'store_id', 'stockout_start_date', 'stockout_end_date', 'stockout_duration_days', 'units_lost', 'revenue_lost', 'resolved_flag']

Sales after cleaning   : 27,450 rows
root
 |-- date: date (nullable = true)
 |-- sku_id: string (nullable = true)
 |-- sku_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- units_sold: integer (nullable =